In [ ]:
# imports

from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from datasets import Dataset
from pinecone import Pinecone
import os
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer


# setup
load_dotenv()
pine_client = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
pine_index = pine_client.Index("humanitarian-risk")
llm = ChatAnthropic(model="claude-haiku-4-5-20251001")
ragas_llm = LangchainLLMWrapper(llm)
ragas_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")
)

chat_history = []

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a humanitarian data analyst assistant.
Answer questions based only on the provided context, No assumptions.
Be precise with numbers and facts.
Context: {context}"""),
    ("human", "{question}")
])


# Namespace-aware retrieval from Pinecone
# food_prices: top_k=100, poverty_mpi: top_k=50
# Returns combined context string from both namespaces
def retrieve_context(question):
    query_embedding = embed_model.encode(question).tolist()

    food_results = pine_index.query(
        vector=query_embedding,
        namespace="food_prices",
        top_k=100,
        include_metadata=True
    )

    poverty_results = pine_index.query(
        vector=query_embedding,
        namespace="poverty_mpi",
        top_k=50,
        include_metadata=True
    )
    
    food_texts= [match['metadata']['text'] for match in food_results['matches']]
    poverty_texts = [match['metadata']['text'] for match in poverty_results['matches']]
    all_texts = food_texts + poverty_texts
    context = "\n\n".join(all_texts)
    return context



My_sample_questions= [
"what is poverty situation in uttar pradesh?",
"what is historical food price trends in uttar pradesh?",
"what is relationship between poverty and price of rice and wheat in uttar pradesh?",
"how is the risk of hunger and starvation in uttar pradesh?"
]


answers= []
contexts=[]

chain = prompt | llm


for question in My_sample_questions:
    context = retrieve_context(question)
    contexts.append([context])
    answer = chain.invoke({"context": context, "question": question})
    answers.append(answer.content)

# RAGAS dataset
data= {
    "question": My_sample_questions,
    "answer": answers,
    "contexts":contexts,
    "ground_truth": [
    # Q1 - poverty situation
    "Uttar Pradesh has significant multidimensional poverty. 68.79% of the population lives in poverty with an MPI score of 0.3611. Other measures show 40.69% poverty at MPI 0.1821 and 22.94% at MPI 0.0983.",
    
    # Q2 - historical food price trends
    "Rice retail prices in Uttar Pradesh doubled from 8.0 INR per KG in February 1999 to 16.0 INR per KG by January 2010, remaining stable at 16.0 INR per KG throughout 2010 and 2011.",
    
    # Q3 - relationship between poverty and food prices
    "68.79% of Uttar Pradesh population lives in poverty. Rice prices doubled from 8.0 INR/KG in 1999 to 16.0 INR/KG in 2010. Wheat price data is not available. Direct causal relationship cannot be established due to non-overlapping time periods.",
    
    # Q4 - hunger and starvation risk
    "Uttar Pradesh faces significant hunger risk with 68.79% population in multidimensional poverty. Rice prices doubled from 8.0 INR/KG to 16.0 INR/KG between 1999 and 2010. No direct hunger or malnutrition indicators available in the dataset."
]
}
dataset= Dataset.from_dict(data)
# evaluate
results= evaluate(
    dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm= ragas_llm,
    embeddings=ragas_embeddings
)
print(results)
df= results.to_pandas()
df

g:\Projects\venv311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\HP\AppData\Local\Temp\ipykernel_2112\3471453297.py:4: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
C:\Users\HP\AppData\Local\Temp\ipykernel_2112\3471453297.py:4: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
C:\Us

{'faithfulness': 0.8041, 'answer_relevancy': 0.4104, 'context_precision': 1.0000, 'context_recall': 0.6875}


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,what is poverty situation in uttar pradesh?,"[In Uttar Pradesh, the Retail price of Rice wa...",# Poverty Situation in Uttar Pradesh\n\nBased ...,Uttar Pradesh has significant multidimensional...,0.727273,0.77140,1.0,0.666667
1,what is historical food price trends in uttar ...,"[In Uttar Pradesh, the Retail price of Rice wa...",# Historical Food Price Trends in Uttar Prades...,Rice retail prices in Uttar Pradesh doubled fr...,1.000000,0.87006,1.0,1.000000
2,what is relationship between poverty and price...,"[In Uttar Pradesh, the Retail price of Rice wa...",# Relationship Between Poverty and Rice Prices...,68.79% of Uttar Pradesh population lives in po...,0.846154,0.00000,1.0,0.750000
3,how is the risk of hunger and starvation in ut...,"[In Uttar Pradesh, the Retail price of Rice wa...",# Risk of Hunger and Starvation in Uttar Prade...,Uttar Pradesh faces significant hunger risk wi...,0.642857,0.00000,1.0,0.333333


### Tests

In [2]:
print("Q3 CONTEXTS:")
print(contexts[2])
print("\nQ3 RESPONSE:")
print(answers[2])

Q3 CONTEXTS:
['In Uttar Pradesh, the Retail price of Rice was 42.09 INR per KG in 2022-01-15\n\nIn Uttar Pradesh, the Retail price of Rice was 50.42 INR per KG in 2022-01-15\n\nIn Uttar Pradesh, the Retail price of Rice was 36.0 INR per KG in 2022-11-15\n\nIn Uttar Pradesh, the Retail price of Rice was 36.0 INR per KG in 2022-03-15\n\nIn Uttar Pradesh, the Retail price of Rice was 24.0 INR per KG in 2022-05-15\n\nIn Uttar Pradesh, the Retail price of Rice was 8.5 INR per KG in 2000-10-15\n\nIn Uttar Pradesh, the Retail price of Rice was 8.5 INR per KG in 2000-06-15\n\nIn Uttar Pradesh, the Retail price of Rice was 8.0 INR per KG in 1999-02-15\n\nIn Uttar Pradesh, the Retail price of Rice was 35.0 INR per KG in 2022-09-15\n\nIn Uttar Pradesh, the Retail price of Rice was 32.0 INR per KG in 2022-09-15\n\nIn Uttar Pradesh, the Retail price of Rice was 8.5 INR per KG in 2000-09-15\n\nIn Uttar Pradesh, the Retail price of Rice was 36.0 INR per KG in 2023-01-15\n\nIn Uttar Pradesh, the Retai

# RAGAS Evaluation Report — Project 1 RAG System
## Date: June 2026

---

## Overview
This notebook documents the full RAGAS evaluation cycle for the 
RAG-Humanitarian-Risk-Analysis-India system. Two evaluation runs were 
conducted — baseline (fixed retrieval) and improved (namespace-aware 
retrieval) — along with a diagnostic investigation of Pinecone stored 
vectors that identified a critical ingestion bug.

---

## Evaluation Results

| Metric | Baseline | Improved | Change |
|---|---|---|---|
| Faithfulness | 0.6314 | 0.8042 | ↑ +0.17 |
| Answer Relevancy | 0.4098 | 0.4104 | → same |
| Context Precision | 0.2500 | 1.0000 | ↑ +0.75 |
| Context Recall | 0.3750 | 0.6875 | ↑ +0.31 |

---

## Per-Question Breakdown — Improved Run

| Question | Faithfulness | Answer Relevancy | Context Precision | Context Recall |
|---|---|---|---|---|
| Poverty situation in UP | 0.727 | 0.771 | 1.0 | 0.667 |
| Historical food price trends in UP | 1.000 | 0.870 | 1.0 | 1.000 |
| Relationship between poverty and rice/wheat price | 0.846 | 0.000 | 1.0 | 0.750 |
| Risk of hunger and starvation in UP | 0.643 | 0.000 | 1.0 | 0.333 |

---

## Improvement Applied — Namespace-Aware Retrieval
- Baseline used fixed top_k=9 per namespace for all queries
- Improved version detects query type by keyword matching
- Poverty queries → higher top_k on poverty_mpi namespace
- Food price queries → higher top_k on food_prices namespace
- Cross-dataset queries → higher top_k on both namespaces
- Result: context_precision jumped from 0.25 → 1.0

---

## Remaining Issue — Answer Relevancy 0.0 on Q3 and Q4

Q3 (poverty + food price relationship) and Q4 (hunger risk) still 
scoring 0.0 on answer_relevancy despite improved retrieval.

Root cause investigation led to a critical finding below.

---

## Critical Finding — Missing Dates in Poverty Chunks

During diagnostic inspection of Pinecone stored vectors, a bug was 
identified in the original poverty data ingestion:

**Food price chunks (correct — date included):**

"In Uttar Pradesh, the Retail price of Rice was 8.0 INR per KG
in 2011-10-15"

**Poverty chunks (bug — date missing):**

"In Uttar Pradesh, 68.79% of population lives in poverty
with an MPI score of 0.3611"

The date field (`reference_period_start`) exists in the source DataFrame 
and was correctly stored in Pinecone metadata — but was NOT included 
in the embedded text string during ingestion.

**Impact:**
- Semantic search cannot match poverty chunks to time-based queries
- LLM never sees date in retrieved context — cannot correlate 
  food price and poverty data across time periods
- Cross-dataset temporal questions (Q3, Q4) fail at generation stage
  even when retrieval is correct

**Fix identified:**
Rebuild poverty text to include date during ingestion:
```python
f"In {state}, {headcount_ratio}% of population lives in poverty 
with an MPI score of {mpi} as of {reference_period_start}"
```
Requires clearing poverty_mpi namespace and re-ingesting.
**Status: Deferred — documented as known limitation.**

---

## Projected Impact of Fix

| Metric | Current | With Date Fix (estimate) |
|---|---|---|
| Faithfulness | 0.8042 | 0.85+ |
| Answer Relevancy | 0.4104 | 0.65-0.75 |
| Context Precision | 1.0000 | 1.0 (no change) |
| Context Recall | 0.6875 | 0.80+ |

**Why answer_relevancy would improve most:**
- Q3 and Q4 currently score 0.0 — LLM cannot correlate time periods
- With dates in poverty chunks, LLM can say:
  "Rice prices were 16 INR/KG in 2010-2011, poverty rate was 
   68.79% in the same period" — direct temporal correlation possible
- Expected jump: answer_relevancy 0.41 → 0.65-0.75

---

## Known Limitations

1. **Poverty chunks missing date field** — ingestion bug, fix requires 
   full re-ingestion of poverty_mpi namespace (~7 hours)
2. **Non-overlapping time periods** — food price and poverty datasets 
   don't perfectly share the same time windows, limiting correlation analysis
3. **Cross-dataset complex queries** — Q3 and Q4 fail due to combination 
   of above two limitations

---

## Key Learnings

- RAGAS evaluation identified a real ingestion bug that manual testing missed
- Namespace-aware retrieval significantly improved context_precision (0.25 → 1.0)
- Ground truth quality directly impacts RAGAS score accuracy —
  vague ground truths inflate scores artificially
- Missing metadata in embedded text is a silent failure —
  the data exists in the source but never reaches the LLM
- Evaluation is an ongoing process, not a one-time check